In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!apt-get update -y
!apt-get install -y tshark
!pip install numpy scipy matplotlib scapy

Mounted at /content/drive
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,998 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [

In [ ]:
import os, sys

%cd /content

if not os.path.exists("/content/Kitsune-py"):
    !git clone https://github.com/ymirsky/Kitsune-py.git

%cd /content/Kitsune-py

!grep -rl "np.Inf" . | xargs sed -i 's/np.Inf/np.inf/g' || true
!grep -rl "numpy.Inf" . | xargs sed -i 's/numpy.Inf/numpy.inf/g' || true
!grep -rl "np.NaN" . | xargs sed -i 's/np.NaN/np.nan/g' || true
!grep -rl "numpy.NaN" . | xargs sed -i 's/numpy.NaN/numpy.nan/g' || true

!sed -i 's#C:\\Program Files\\Wireshark\\\\tshark.exe#tshark#g' FeatureExtractor.py

sys.path.append("/content/Kitsune-py")
sys.path.append("/content/Kitsune-py/KitNET")

from Kitsune import Kitsune

print("Kitsune import successful")

/content
Cloning into 'Kitsune-py'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 82 (delta 25), reused 23 (delta 22), pack-reused 39 (from 1)
Receiving objects: 100% (82/82), 25.65 MiB | 23.22 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/Kitsune-py
sed: no input files
sed: no input files
Importing Scapy Library
Kitsune import successful


In [9]:
BASE_DIR = "/content/drive/MyDrive/kitsune_exp"
os.makedirs(BASE_DIR, exist_ok=True)

TRAIN_PCAP = "/content/dump_00001_20200113152847.pcap"
CLEAN_TEST_PCAP = "/content/dump_00002_20200113162614.pcap"
MAL_TEST_PCAP = "/content/dns2tcp_tunnel_1111_doh10_2020-03-31T21_54_36.738995.pcap"

MAX_AE = 10
PACKET_LIMIT = 1200000
FM_GRACE = 30000
AD_GRACE = 800000

CHECKPOINT_EVERY = 10000
THRESHOLD_PERCENTILE = 99.0
BUFFER_PACKETS = 1000

In [10]:
import pickle
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm


def get_detector(K):
    for name in ["AnomDetector", "AD", "kitnet", "KitNET"]:
        if hasattr(K, name):
            return getattr(K, name), name
    print(K.__dict__.keys())
    raise AttributeError("Could not find KitNET detector inside Kitsune object.")


def set_detector(K, detector, attr_name):
    setattr(K, attr_name, detector)


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def plot_kitsune_logprob_scatter(rmse, fm_grace, ad_grace, output_path):
    rmse = np.asarray(rmse)
    eps = 1e-12

    exec_start = fm_grace + ad_grace + 1

    benign_sample = rmse[exec_start:]
    benign_sample = benign_sample[np.isfinite(benign_sample)]
    benign_sample = benign_sample[benign_sample > 0]

    if len(benign_sample) < 10:
        benign_sample = rmse[fm_grace:]
        benign_sample = benign_sample[np.isfinite(benign_sample)]
        benign_sample = benign_sample[benign_sample > 0]

    log_benign = np.log(np.maximum(benign_sample, eps))

    log_probs = norm.logsf(
        np.log(np.maximum(rmse, eps)),
        np.mean(log_benign),
        np.std(log_benign) + eps
    )

    x = np.arange(exec_start, len(rmse))

    plt.figure(figsize=(10, 5))
    fig = plt.scatter(
        x,
        rmse[exec_start:],
        s=0.5,
        c=log_probs[exec_start:],
        cmap="RdYlGn"
    )

    plt.yscale("log")
    plt.title("Kitsune Anomaly Scores from Execution Phase")
    plt.ylabel("RMSE (log scaled)")
    plt.xlabel("Packet index")

    figbar = plt.colorbar(fig)
    figbar.ax.set_ylabel("Log Probability", rotation=270, labelpad=15)

    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_rmse_curve(rmse, threshold, output_path, title, buffer_packets=None):
    plt.figure(figsize=(10, 5))
    plt.plot(rmse, linewidth=0.6, label="RMSE")
    plt.axhline(threshold, linestyle="--", label=f"Threshold={threshold:.6f}")

    if buffer_packets is not None:
        plt.axvline(buffer_packets, linestyle="--", label="Buffer ends")

    plt.yscale("log")
    plt.xlabel("Packet index")
    plt.ylabel("RMSE")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_colored_rmse_curve(
    rmse,
    threshold,
    output_path,
    title,
    buffer_packets=None,
    reference_rmse=None,
):
    rmse = np.asarray(rmse)
    eps = 1e-12

    # Use reference distribution to estimate probabilities
    if reference_rmse is None:
        reference_rmse = rmse

    reference_rmse = np.asarray(reference_rmse)
    reference_rmse = reference_rmse[np.isfinite(reference_rmse)]
    reference_rmse = reference_rmse[reference_rmse > 0]

    log_ref = np.log(np.maximum(reference_rmse, eps))

    log_probs = norm.logsf(
        np.log(np.maximum(rmse, eps)),
        np.mean(log_ref),
        np.std(log_ref) + eps
    )

    x = np.arange(len(rmse))

    plt.figure(figsize=(12, 5))

    scatter = plt.scatter(
        x,
        rmse,
        s=1,
        c=log_probs,
        cmap="RdYlGn",
    )

    plt.axhline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Threshold={threshold:.6f}"
    )

    if buffer_packets is not None:
        plt.axvline(
            buffer_packets,
            linestyle="--",
            linewidth=2,
            label="Buffer ends"
        )

    plt.yscale("log")

    plt.xlabel("Packet index")
    plt.ylabel("RMSE (log scale)")
    plt.title(title)

    plt.legend()

    cbar = plt.colorbar(scatter)
    cbar.ax.set_ylabel("Log Probability", rotation=270, labelpad=15)

    plt.grid(True)

    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_rmse_distribution(rmse, threshold, output_path, title):
    rmse = np.asarray(rmse)
    rmse = rmse[np.isfinite(rmse)]

    plt.figure(figsize=(8, 5))
    plt.hist(rmse, bins=200, density=True, alpha=0.7)
    plt.axvline(threshold, linestyle="--", label=f"Threshold={threshold:.6f}")
    plt.xlabel("RMSE")
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_overlay(clean_rmse, mal_rmse, threshold, output_path):
    clean_rmse = np.asarray(clean_rmse)
    mal_rmse = np.asarray(mal_rmse)

    plt.figure(figsize=(8, 5))
    plt.hist(clean_rmse, bins=200, density=True, alpha=0.5, label="Clean")
    plt.hist(mal_rmse, bins=200, density=True, alpha=0.5, label="Malicious")
    plt.axvline(threshold, linestyle="--", label=f"Threshold={threshold:.6f}")
    plt.xlabel("RMSE")
    plt.ylabel("Density")
    plt.title("Kitsune RMSE Distribution: Clean vs Malicious")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_train_clean_overlay(train_rmse, clean_rmse, threshold, output_path):
    train_eval = train_rmse[FM_GRACE + AD_GRACE:]
    if len(train_eval) < 10:
        train_eval = train_rmse[FM_GRACE:]

    clean_eval = clean_rmse[BUFFER_PACKETS:]

    plt.figure(figsize=(8, 5))
    plt.hist(train_eval, bins=200, density=True, alpha=0.5, label="Train PCAP")
    plt.hist(clean_eval, bins=200, density=True, alpha=0.5, label="Clean Test PCAP")
    plt.axvline(threshold, linestyle="--", label=f"Threshold={threshold:.6f}")
    plt.xlabel("RMSE")
    plt.ylabel("Density")
    plt.title("Kitsune RMSE Distribution: Train vs Clean Test")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

In [11]:
OUT_TRAIN = os.path.join(BASE_DIR, "train")
CKPT_DIR = os.path.join(OUT_TRAIN, "checkpoints")
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

K = Kitsune(
    TRAIN_PCAP,
    PACKET_LIMIT,
    MAX_AE,
    FM_GRACE,
    AD_GRACE
)

rmse_train = []
start = time.time()

i = 0
while True:
    i += 1
    rmse = K.proc_next_packet()

    if rmse == -1:
        break

    rmse_train.append(float(rmse))

    if i % 1000 == 0:
        print("Processed", i, "packets")

    if CHECKPOINT_EVERY > 0 and i % CHECKPOINT_EVERY == 0:
        detector, detector_attr = get_detector(K)
        ckpt_path = os.path.join(CKPT_DIR, f"kitnet_detector_{i}.pkl")
        save_pickle(detector, ckpt_path)
        print("Saved checkpoint:", ckpt_path)

elapsed = time.time() - start
rmse_train = np.asarray(rmse_train)

np.save(os.path.join(OUT_TRAIN, "train_rmse.npy"), rmse_train)

detector, detector_attr = get_detector(K)
detector_path = os.path.join(OUT_TRAIN, "kitnet_detector_final.pkl")
save_pickle(detector, detector_path)

print("Detector attribute name:", detector_attr)
print("Saved detector:", detector_path)
print("Training elapsed:", elapsed)

Parsing with tshark...
tshark parsing complete. File saved as: /content/dump_00001_20200113152847.pcap.tsv
counting lines in file...
There are 1189065 Packets.
Feature-Mapper: train-mode, Anomaly-Detector: off-mode
Processed 1000 packets
Processed 2000 packets
Processed 3000 packets
Processed 4000 packets
Processed 5000 packets
Processed 6000 packets
Processed 7000 packets
Processed 8000 packets
Processed 9000 packets
Processed 10000 packets
Saved checkpoint: /content/drive/MyDrive/kitsune_exp/train/checkpoints/kitnet_detector_10000.pkl
Processed 11000 packets
Processed 12000 packets
Processed 13000 packets
Processed 14000 packets
Processed 15000 packets
Processed 16000 packets
Processed 17000 packets
Processed 18000 packets
Processed 19000 packets
Processed 20000 packets
Saved checkpoint: /content/drive/MyDrive/kitsune_exp/train/checkpoints/kitnet_detector_20000.pkl
Processed 21000 packets
Processed 22000 packets
Processed 23000 packets
Processed 24000 packets
Processed 25000 packets


In [12]:
execution_start = FM_GRACE + AD_GRACE

if len(rmse_train) > execution_start + 10:
    threshold_sample = rmse_train[execution_start:]
else:
    threshold_sample = rmse_train[max(FM_GRACE, 0):]

threshold_sample = threshold_sample[np.isfinite(threshold_sample)]
threshold_sample = threshold_sample[threshold_sample > 0]

threshold = float(np.percentile(threshold_sample, THRESHOLD_PERCENTILE))

threshold_info = {
    "threshold_rmse": threshold,
    "threshold_percentile": THRESHOLD_PERCENTILE,
    "num_threshold_samples": int(len(threshold_sample)),
    "fm_grace": FM_GRACE,
    "ad_grace": AD_GRACE,
    "packet_limit": PACKET_LIMIT,
    "detector_attr": detector_attr,
    "detector_path": detector_path,
}

with open(os.path.join(OUT_TRAIN, "threshold.json"), "w") as f:
    json.dump(threshold_info, f, indent=4)

plot_rmse_curve(
    rmse_train,
    threshold,
    os.path.join(OUT_TRAIN, "train_rmse_curve.png"),
    "Kitsune Training RMSE"
)

plot_colored_rmse_curve(
    rmse_train,
    threshold,
    os.path.join(OUT_TRAIN, "train_rmse_curve_colored.png"),
    "Kitsune Training RMSE",
    reference_rmse=threshold_sample
)

plot_rmse_distribution(
    rmse_train,
    threshold,
    os.path.join(OUT_TRAIN, "train_rmse_distribution.png"),
    "Kitsune Training RMSE Distribution"
)

print("Threshold:", threshold)
print("Saved training plots to:", OUT_TRAIN)

plot_kitsune_logprob_scatter(
    rmse_train,
    FM_GRACE,
    AD_GRACE,
    os.path.join(OUT_TRAIN, "train_kitsune_logprob_scatter.png")
)

/tmp/ipykernel_4687/985441213.py:160: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(output_path, dpi=300, bbox_inches="tight")


Threshold: 0.11595988820485163
Saved training plots to: /content/drive/MyDrive/kitsune_exp/train


In [ ]:
def run_kitsune_test(test_pcap, label_name, detector_path, detector_attr, threshold, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    detector = load_pickle(detector_path)

    K_test = Kitsune(
        test_pcap,
        PACKET_LIMIT,
        MAX_AE,
        0,
        0
    )

    set_detector(K_test, detector, detector_attr)

    rmse_scores = []
    preds = []

    i = 0
    start = time.time()

    while True:
        i += 1
        rmse = K_test.proc_next_packet()

        if rmse == -1:
            break

        rmse = float(rmse)
        rmse_scores.append(rmse)

        if i <= BUFFER_PACKETS:
            preds.append(0)
        else:
            preds.append(int(rmse > threshold))

        if i % 1000 == 0:
            print(f"{label_name}: processed {i} packets")

    elapsed = time.time() - start

    rmse_scores = np.asarray(rmse_scores)
    preds = np.asarray(preds)

    np.save(os.path.join(output_dir, f"{label_name}_rmse.npy"), rmse_scores)
    np.save(os.path.join(output_dir, f"{label_name}_preds.npy"), preds)

    eval_preds = preds[BUFFER_PACKETS:] if len(preds) > BUFFER_PACKETS else np.asarray([])
    anomaly_fraction = float(np.mean(eval_preds)) if len(eval_preds) > 0 else 0.0

    summary = {
        "label": label_name,
        "pcap": test_pcap,
        "num_packets": int(len(rmse_scores)),
        "buffer_packets": BUFFER_PACKETS,
        "threshold": float(threshold),
        "anomaly_fraction_after_buffer": anomaly_fraction,
        "elapsed_sec": float(elapsed),
        "rmse_mean": float(np.mean(rmse_scores)) if len(rmse_scores) else None,
        "rmse_median": float(np.median(rmse_scores)) if len(rmse_scores) else None,
        "rmse_p95": float(np.percentile(rmse_scores, 95)) if len(rmse_scores) else None,
        "rmse_p99": float(np.percentile(rmse_scores, 99)) if len(rmse_scores) else None,
    }

    with open(os.path.join(output_dir, f"{label_name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=4)

    plot_rmse_curve(
        rmse_scores,
        threshold,
        os.path.join(output_dir, f"{label_name}_rmse_curve.png"),
        f"Kitsune Test RMSE: {label_name}",
        buffer_packets=BUFFER_PACKETS
    )

    plot_colored_rmse_curve(
        rmse_scores,
        threshold,
        os.path.join(output_dir, f"{label_name}_rmse_curve_colored.png"),
        f"Kitsune Test RMSE: {label_name}",
        buffer_packets=BUFFER_PACKETS,
        reference_rmse=threshold_sample
    )

    plot_rmse_distribution(
        rmse_scores,
        threshold,
        os.path.join(output_dir, f"{label_name}_rmse_distribution.png"),
        f"Kitsune RMSE Distribution: {label_name}"
    )

    print(label_name, "summary:", summary)

    return rmse_scores, preds, summary

In [14]:
OUT_CLEAN = os.path.join(BASE_DIR, "test_clean")
OUT_MAL = os.path.join(BASE_DIR, "test_malicious")

clean_rmse, clean_preds, clean_summary = run_kitsune_test(
    CLEAN_TEST_PCAP,
    "clean",
    detector_path,
    detector_attr,
    threshold,
    OUT_CLEAN
)

mal_rmse, mal_preds, mal_summary = run_kitsune_test(
    MAL_TEST_PCAP,
    "malicious",
    detector_path,
    detector_attr,
    threshold,
    OUT_MAL
)

plot_overlay(
    clean_rmse[BUFFER_PACKETS:],
    mal_rmse[BUFFER_PACKETS:],
    threshold,
    os.path.join(BASE_DIR, "rmse_overlay_clean_vs_malicious.png")
)

plot_train_clean_overlay(
    rmse_train,
    clean_rmse,
    threshold,
    os.path.join(BASE_DIR, "rmse_overlay_train_vs_clean.png")
)

print("Saved all results to:", BASE_DIR)

Parsing with tshark...
tshark parsing complete. File saved as: /content/dump_00002_20200113162614.pcap.tsv
counting lines in file...
There are 1273144 Packets.
Feature-Mapper: train-mode, Anomaly-Detector: off-mode
clean: processed 1000 packets
clean: processed 2000 packets
clean: processed 3000 packets
clean: processed 4000 packets
clean: processed 5000 packets
clean: processed 6000 packets
clean: processed 7000 packets
clean: processed 8000 packets
clean: processed 9000 packets
clean: processed 10000 packets
clean: processed 11000 packets
clean: processed 12000 packets
clean: processed 13000 packets
clean: processed 14000 packets
clean: processed 15000 packets
clean: processed 16000 packets
clean: processed 17000 packets
clean: processed 18000 packets
clean: processed 19000 packets
clean: processed 20000 packets
clean: processed 21000 packets
clean: processed 22000 packets
clean: processed 23000 packets
clean: processed 24000 packets
clean: processed 25000 packets
clean: processed 26

/tmp/ipykernel_4687/985441213.py:90: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(output_path, dpi=300, bbox_inches="tight")
/tmp/ipykernel_4687/985441213.py:160: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(output_path, dpi=300, bbox_inches="tight")


clean summary: {'label': 'clean', 'pcap': '/content/dump_00002_20200113162614.pcap', 'num_packets': 1200000, 'buffer_packets': 1000, 'threshold': 0.11595988820485163, 'anomaly_fraction_after_buffer': 0.02554628857381151, 'elapsed_sec': 16039.24439907074, 'rmse_mean': 1.5397404065041784, 'rmse_median': 0.009947521049701098, 'rmse_p95': 0.11151616650731667, 'rmse_p99': 0.11991416977645657}
Parsing with tshark...
tshark parsing complete. File saved as: /content/dns2tcp_tunnel_1111_doh10_2020-03-31T21_54_36.738995.pcap.tsv
counting lines in file...
There are 4923 Packets.
Feature-Mapper: train-mode, Anomaly-Detector: off-mode
malicious: processed 1000 packets
malicious: processed 2000 packets
malicious: processed 3000 packets
malicious: processed 4000 packets
malicious summary: {'label': 'malicious', 'pcap': '/content/dns2tcp_tunnel_1111_doh10_2020-03-31T21_54_36.738995.pcap', 'num_packets': 4922, 'buffer_packets': 1000, 'threshold': 0.11595988820485163, 'anomaly_fraction_after_buffer': 0.